<h2>Potential Talents: NLP Project

In [43]:
from pathlib import Path
PROJ_ROOT = Path().resolve().parents[0]
DATA_DIR = PROJ_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
RAW_DATA_PATH = RAW_DATA_DIR / "potential-talents - Aspiring human resources - seeking human resources.csv"

In [44]:
import pandas as pd

In [54]:
data = pd.read_csv(RAW_DATA_PATH)
data.head(5)

,id,job_title,location,connection,fit
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85,NaN
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500+,NaN
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,NaN
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+,NaN
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+,NaN


In [46]:
data.shape

(104, 5)

In [47]:
# Remove duplicates, then rebuild a continuous id column
data = data.drop_duplicates(subset=["job_title", "location", "connection", "fit"]).reset_index(drop=True) #All id's are unique, but the info in the other columns (when combined), aren't
data["id"] = data.index + 1

data.shape

(53, 5)

In [48]:
print(data.loc[0, 'job_title'])


2019 C.T. Bauer College of Business Graduate (Magna Cum Laude) and aspiring Human Resources professional


In terms of exploratory data analysis (EDA) the 'job_title' and 'location' features show high cardinality upon initial visual inspection. 'connection' and 'fit' will be the most useful in the case of EDA:

In [49]:
data.nunique()

id            53
job_title     52
location      41
connection    33
fit            0
dtype: int64

<h4>Steps Required to Create an NLP Recommender System:

1. Define the 'ideal' candidate via a new description or by 'starring' a candidate in the list.
2. Tokenize.
3. Compute the TF-IDF vectors. Converts the words into a high dimension vector.
4. Compute cosine similarity (0 - 1) between the 'ideal' candidate and the existing candidates.
5. Rank the cosine similarity.

In [50]:
# First we shall combine all the features (apart from 'id') into a single string to capture as much info as possible:
data["combined"] = (data["job_title"].astype(str)+ " located in " + data["location"].astype(str) + ", with " + data["connection"].astype(str)+ " connections."
)

# Test
data["combined"][0]

'2019 C.T. Bauer College of Business Graduate (Magna Cum Laude) and aspiring Human Resources professional located in Houston, Texas, with 85 connections.'

In [51]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Next we transform the "combined" features into the input required for cosine similarity

data_by_id = data.set_index("id", drop=False)

# Tokenization + TF-IDF Vectorization:
vectorizer = TfidfVectorizer(stop_words="english") # Carries out both tokenization and vectorization
tfidf_matrix = vectorizer.fit_transform(data_by_id["combined"].fillna(""))

# Row order in tfidf_matrix matches data_by_id.index, making each vector traceable to its id
candidate_ids = data_by_id.index.to_numpy()

In [59]:
from sklearn.metrics.pairwise import cosine_similarity

# Way to 'star' a candidate, then calculate cosine similarity.

def recommend_candidates_id(base_data, starred_id, top_n=10):
    """
    Recommends candidates based on similarity to a starred candidate by selecting the 'id' of the starred candidate.
    """
    base = base_data.copy()

    if "combined" not in base.columns:
        required_cols = ["job_title", "location", "connection"]
        missing = [col for col in required_cols if col not in base.columns]
        if missing:
            raise ValueError(f"Missing required columns to build 'combined': {missing}")

        base["combined"] = (
            base["job_title"].astype(str)
            + " located in "
            + base["location"].astype(str)
            + ", with "
            + base["connection"].astype(str)
            + " connections."
        )

    if "id" in base.columns:
        base_by_id = base.set_index("id", drop=False)
    else:
        base_by_id = base.copy()

    if starred_id not in base_by_id.index:
        raise ValueError(f"id {starred_id} not found in base_data")

    matrix = vectorizer.fit_transform(base_by_id["combined"].fillna(""))

    starred_idx = base_by_id.index.get_loc(starred_id)
    scores = cosine_similarity(matrix[starred_idx], matrix).ravel()

    ranked = base_by_id.copy()
    ranked["fit"] = scores
    ranked = ranked[ranked["fit"] > 0].sort_values("fit", ascending=False)

    return ranked.head(top_n)

# example:
recommend_candidates_id(data, 30, top_n=10)


,id,job_title,location,connection,fit,combined
id,,,,,,
28,28,Seeking Human Resources Opportunities,"Chicago, Illinois",390,1.000000,Seeking Human Resources Opportunities located ...
30,30,Seeking Human Resources Opportunities,"Chicago, Illinois",390,1.000000,Seeking Human Resources Opportunities located ...
91,91,Lead Official at Western Illinois University,Greater Chicago Area,39,0.326215,Lead Official at Western Illinois University l...
94,94,Seeking Human Resources Opportunities. Open t...,Amerika Birleşik Devletleri,415,0.240282,Seeking Human Resources Opportunities. Open t...
73,73,"Aspiring Human Resources Manager, seeking inte...","Houston, Texas Area",7,0.231110,"Aspiring Human Resources Manager, seeking inte..."
92,92,Seeking employment opportunities within Custom...,"Torrance, California",64,0.195782,Seeking employment opportunities within Custom...
29,29,Aspiring Human Resources Management student se...,"Houston, Texas Area",500+,0.178017,Aspiring Human Resources Management student se...
27,27,Aspiring Human Resources Management student se...,"Houston, Texas Area",500+,0.178017,Aspiring Human Resources Management student se...
40,40,Seeking Human Resources HRIS and Generalist Po...,Greater Philadelphia Area,500+,0.172474,Seeking Human Resources HRIS and Generalist Po...


In [60]:
# We could also use a description of an ideal candidate to act as the benchmark 'ideal' candidate:

ideal_candidate_description = "A business graduate"

def recommend_candidates_description(base_data, ideal_candidate_description, top_n=10):
    """
    Recommends candidates based on similarity to an ideal candidate description.
    Only non-zero cosine similarities are returned.
    """
    if "id" in base_data.columns:
        base_by_id = base_data.set_index("id", drop=False)
    else:
        base_by_id = base_data.copy()

    matrix = vectorizer.fit_transform(base_by_id["combined"].fillna(""))
    ideal_vector = vectorizer.transform([ideal_candidate_description])

    scores = cosine_similarity(ideal_vector, matrix).ravel()

    ranked = base_by_id.copy()
    ranked["fit"] = scores
    ranked = ranked[ranked["fit"] > 0].sort_values("fit", ascending=False)

    return ranked.head(top_n)

recommend_candidates_description(data_by_id, ideal_candidate_description)


,id,job_title,location,connection,fit,combined
id,,,,,,
1,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85,0.387305,2019 C.T. Bauer College of Business Graduate (...
21,21,Business Management Major and Aspiring Human R...,"Monroe, Louisiana Area",5,0.201242,Business Management Major and Aspiring Human R...
51,51,Business Intelligence and Analytics at Travelers,Greater New York City Area,49,0.167003,Business Intelligence and Analytics at Travele...
30,30,Senior Human Resources Business Partner at Hei...,"Chattanooga, Tennessee Area",455,0.158494,Senior Human Resources Business Partner at Hei...
45,45,Student at Indiana University Kokomo - Busines...,"Lafayette, Indiana",19,0.125188,Student at Indiana University Kokomo - Busines...


In [61]:
# Next is to implement these functions into a front end